# Loans Data

This project looks to explore financial lending data from [Lending Club](https://www.lendingclub.com/), a marketplace for personal loans that matches borrowers with investors. The Lending Club's website lists approved loans. Qualified investors can view the borrower's credit score, the purpose of the loan, and other details in the loan applications. Once a lender is ready to back a loan, it selects the amount of money it wants to lend. When the loan amount the borrower requested is fully funded, the borrower receives the money, minus the origination fee that Lending Club charges.

In the project specifically, my aim is to practice optimisation using DataFrames and utilising batch processing.

## Exploring Loans Data

In [1]:
import pandas as pd
import numpy as np

import warnings                                                     # This section is embedded to hide additional error messages which are otherwise
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)  # expected for this exercise.

loans = pd.read_csv('loans_2007.csv', nrows = 5)
loans

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,1077501,1296599.0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,...,171.62,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
1,1077430,1314167.0,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,...,119.66,Sep-2013,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
2,1077175,1313524.0,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,...,649.91,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
3,1076863,1277178.0,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,...,357.48,Apr-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
4,1075358,1311748.0,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,...,67.79,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0


As noted, we are using the `loans_2007.csv` dataset, and upon initial load, we have selected the first 5 rows to help visualise and understand the data. Inherently, this will not cover all potential data quality concerns, but this approach at least let's us understand the data types (i.e., `str`, `int`, `float` dtypes etc.).

Next, to load in the entire dataset, we are going to load in the dataset in chunks, such that each chunk is approximately 5 MB. To allow a general use case, we will take multiple initial chunk segments, till we can identify when we cross the 5 MB threshold.

In [2]:
chunk_size_MB = 0
chunk_size = 500

while chunk_size_MB < 5.0:
    temp = pd.read_csv('loans_2007.csv', nrows = chunk_size)
    chunk_size_MB = temp.memory_usage(deep=True).sum() / 1024 ** 2
    chunk_size += 100

chunk_size -= 100
loans = pd.read_csv('loans_2007.csv', chunksize = chunk_size)

chunk_size_MB = []
for chunk in loans:
    chunk_size_MB.append(chunk.memory_usage(deep=True).sum()/1024 ** 2)

print(f"The average memory usage of each loaded chunk is {sum(chunk_size_MB)/len(chunk_size_MB):.2f} MB")
print(f"The total memory is {sum(chunk_size_MB):.2f} MB")

original_memory_footprint = sum(chunk_size_MB) # We will store this value for later comparison after we have implemented some optimisations.

The average memory usage of each loaded chunk is 4.70 MB
The total memory is 23.48 MB


To explore the data, which is now in chunks, we are going to explore these three areas (for each chunk):

- How many columns have a numeric type? How many have a string type?
- How many unique values are there in each string column? How many of the string columns contain values that are less than 50% unique?
- Which float columns have no missing values and could be candidates for conversion to the integer type?

### Column Types

Below, we will explore the first question: _How many columns have a numeric type? How many have a string type?_

In [3]:
numeric = []
string = []

loans = pd.read_csv('loans_2007.csv', chunksize = chunk_size)

for chunk in loans:
    num_col = chunk.select_dtypes(include=[np.number]).shape[1]
    numeric.append(num_col)
    str_col = chunk.select_dtypes(include=['object', 'str']).shape[1]
    string.append(str_col)

It appears that the majority of the columns are numeric, with only a few being string columns. Further, when looking at the data, it appears that there is consistency across most of our chunks; however, in the last two chunks, there is a slight increase in the number of string columns. This is likely indicative that one column, which was previously considered a numeric column, is no longer being considered as such. We will explore what columns this is.

In [4]:
loans = pd.read_csv('loans_2007.csv', chunksize=chunk_size)

def consistency_check(data):
    i = 0
    str_col = []
    num_col = []

    for chunk in data:
        chunk_obj_cols = chunk.select_dtypes(include=['object','str']).columns.tolist()
        str_col.extend(chunk_obj_cols)
        chunk_obj_cols = chunk.select_dtypes(include=[np.number]).columns.tolist()
        num_col.extend(chunk_obj_cols)
        i += 1

    str_col_count = pd.Series(str_col).value_counts()
    num_col_count = pd.Series(num_col).value_counts()

    inconsistent_str = str_col_count[str_col_count != i].rename_axis('columnName')
    inconsistent_num = num_col_count[num_col_count != i].rename_axis('columnName')

    if not inconsistent_str.empty:
        print(f"The inconsistent columns are:\n"
              f"\n"
              f"String Columns:\n"
              f"{str(inconsistent_str)}\n"
              f"\n"
              f"Numeric Columns:\n"
              f"{str(inconsistent_num)}\n")
    else:
        print(f"The columns are consistent.")

consistency_check(loans)

The inconsistent columns are:

String Columns:
columnName
id    1
Name: count, dtype: int64

Numeric Columns:
columnName
id    4
Name: count, dtype: int64



Based on the executed code above, we can see that there is one columns that are inconsistent across the chunks. The "id" column is a numeric column in most chunks, but a string column in two chunks. This suggests that there may be some data quality issues with these columns, and further investigation may be needed to determine the root cause of the inconsistency. We can check by attempting to force load the column as `int64` type, and identifying any "error" rows in the data set.

In [5]:
bad_rows = []
for chunk in pd.read_csv('loans_2007.csv', chunksize=chunk_size, dtype=str):
    bad = chunk[pd.to_numeric(chunk['id'], errors='coerce').isna()]
    bad_rows.append(bad)

bad_rows = pd.concat(bad_rows)
bad_rows

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
39786,Loans that do not meet the credit policy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42536,Total amount funded in policy code 1: 471701350,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42537,Total amount funded in policy code 2: 0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Based on the executed code above, we can identify there are three rows which have 'bad' IDs. Given the remainder of the row is marked as `NaN`, we can reasonably determine these are spacers within the file, and we can safely ignore them. We will now attempt to reload the file, skipping the "error" rows in the data set, and rerun our inconsistent row checks to see if there are any mismatched column types.

In [6]:
bad_idx = bad_rows.index.tolist()
skip = [n + 1 for n in bad_idx]

loans = pd.read_csv('loans_2007.csv', chunksize=chunk_size, skiprows=skip)
consistency_check(loans)

The columns are consistent.


### Categorisation

We will now explore the second question: _How many unique values are there in each string column? How many of the string columns contain values that are less than 50 distinct unique values?_

In [7]:
uniques = {}

loans = pd.read_csv('loans_2007.csv', chunksize=chunk_size, skiprows=skip)

for chunk in loans:
    strings_only = chunk.select_dtypes(include=['object', 'str'])
    col_idx = strings_only.columns
    for col in col_idx:
        count = strings_only[col].value_counts()
        if col in uniques:
            uniques[col].append(count)
        else:
            uniques[col] = [count]

unique_combined = {}
unique_stats = {
    'column_name': [],
    'total_values': [],
    'unique_values': [],
}

for col in uniques:
    col_concat = pd.concat(uniques[col])
    col_grouped = col_concat.groupby(col_concat.index).sum()
    unique_combined[col] = col_grouped

    unique_stats['column_name'].append(col)
    unique_stats['total_values'].append(col_grouped.sum())
    unique_stats['unique_values'].append(col_grouped.shape[0])

stats = pd.DataFrame(unique_stats)
stats['unique_ratio'] = stats['unique_values'] / stats['total_values']
stats = stats.sort_values('unique_ratio', ascending=True)
stats

,column_name,total_values,unique_values,unique_ratio
20,application_type,42535,1,0.000024
17,initial_list_status,42535,1,0.000024
0,term,42535,2,0.000047
10,pymnt_plan,42535,2,0.000047
7,verification_status,42535,3,0.000071
6,home_ownership,42535,5,0.000118
2,grade,42535,7,0.000165
9,loan_status,42535,9,0.000212
5,emp_length,41423,11,0.000266
11,purpose,42535,14,0.000329


Based on the table above, we can see there are approximately 12 columns with 50 or less distinct unique values. They appear to be candidates for categorisation, and may be worth exploring further. We will explore these columns in a later section, but based on a preliminary analysis of column headers, it appears some may be better categorised as `datetime` objects. For now, we will move onto the last question.

### Float Columns Candidates for Integer Conversion

Below we will explore the last question: _Which float columns have no missing values and could be candidates for conversion to the integer type?_

In [8]:
missing = []

loans = pd.read_csv('loans_2007.csv', chunksize=chunk_size, skiprows=skip)

for chunk in loans:
    floats = chunk.select_dtypes(include=['float'])
    missing.append(floats.apply(pd.isnull).sum())

combined_missing = pd.concat(missing)
summed_missing = combined_missing.groupby(combined_missing.index).sum().sort_values()
summed_missing

collection_recovery_fee          0
dti                              0
loan_amnt                        0
member_id                        0
last_pymnt_amnt                  0
installment                      0
funded_amnt_inv                  0
funded_amnt                      0
total_pymnt                      0
total_pymnt_inv                  0
total_rec_int                    0
revol_bal                        0
recoveries                       0
policy_code                      0
out_prncp_inv                    0
out_prncp                        0
total_rec_prncp                  0
total_rec_late_fee               0
annual_inc                       4
open_acc                        29
delinq_amnt                     29
delinq_2yrs                     29
acc_now_delinq                  29
inq_last_6mths                  29
pub_rec                         29
total_acc                       29
tax_liens                      105
collections_12_mths_ex_med     145
chargeoff_within_12_

## Optimising String Columns

We can achieve the greatest memory improvements by converting the `string` columns to `numeric` types. We will achieve this by doing the following:

1. Converting all the columns where the values are less than 50 unique values to the category type, and the columns that contain numeric values to the float type.
2. Converting all the columns where there are datetime values to the `datetime` type.
3. Inspecting the columns that contain numeric values to determine if they can be converted to the a `float` or `int` type (e.g., int_rate).

We already prepared a variable `stats` which contains a list of our string columns, including value counts. We can use it as indicator for investigation, before we commit to converting any of them.

In [9]:
investigation_columns = stats['column_name']

invesigation_load = pd.read_csv(
    'loans_2007.csv',
    skiprows=skip,
    usecols=investigation_columns,
    nrows=10
)

invesigation_load

,term,int_rate,grade,sub_grade,emp_title,emp_length,home_ownership,verification_status,issue_d,loan_status,...,purpose,title,zip_code,addr_state,earliest_cr_line,revol_util,initial_list_status,last_pymnt_d,last_credit_pull_d,application_type
0,36 months,10.65%,B,B2,NaN,10+ years,RENT,Verified,Dec-2011,Fully Paid,...,credit_card,Computer,860xx,AZ,Jan-1985,83.7%,f,Jan-2015,Jun-2016,INDIVIDUAL
1,60 months,15.27%,C,C4,Ryder,< 1 year,RENT,Source Verified,Dec-2011,Charged Off,...,car,bike,309xx,GA,Apr-1999,9.4%,f,Apr-2013,Sep-2013,INDIVIDUAL
2,36 months,15.96%,C,C5,NaN,10+ years,RENT,Not Verified,Dec-2011,Fully Paid,...,small_business,real estate business,606xx,IL,Nov-2001,98.5%,f,Jun-2014,Jun-2016,INDIVIDUAL
3,36 months,13.49%,C,C1,AIR RESOURCES BOARD,10+ years,RENT,Source Verified,Dec-2011,Fully Paid,...,other,personel,917xx,CA,Feb-1996,21%,f,Jan-2015,Apr-2016,INDIVIDUAL
4,60 months,12.69%,B,B5,University Medical Group,1 year,RENT,Source Verified,Dec-2011,Current,...,other,Personal,972xx,OR,Jan-1996,53.9%,f,Jun-2016,Jun-2016,INDIVIDUAL
5,36 months,7.90%,A,A4,Veolia Transportaton,3 years,RENT,Source Verified,Dec-2011,Fully Paid,...,wedding,My wedding loan I promise to pay back,852xx,AZ,Nov-2004,28.3%,f,Jan-2015,Jan-2016,INDIVIDUAL
6,60 months,15.96%,C,C5,Southern Star Photography,8 years,RENT,Not Verified,Dec-2011,Fully Paid,...,debt_consolidation,Loan,280xx,NC,Jul-2005,85.6%,f,May-2016,May-2016,INDIVIDUAL
7,36 months,18.64%,E,E1,MKC Accounting,9 years,RENT,Source Verified,Dec-2011,Fully Paid,...,car,Car Downpayment,900xx,CA,Jan-2007,87.5%,f,Jan-2015,Dec-2014,INDIVIDUAL
8,60 months,21.28%,F,F2,NaN,4 years,OWN,Source Verified,Dec-2011,Charged Off,...,small_business,Expand Business & Buy Debt Portfolio,958xx,CA,Apr-2004,32.6%,f,Apr-2012,Aug-2012,INDIVIDUAL
9,60 months,12.69%,B,B5,Starbucks,< 1 year,RENT,Verified,Dec-2011,Charged Off,...,other,Building my credit history.,774xx,TX,Sep-2004,36.5%,f,Nov-2012,Mar-2013,INDIVIDUAL


Based on this initial load, we can determine which columns are suitable for conversion:

- **Categorisation**
    - grade
    - sub_grade
    - emp_length
    - home_ownership
    - verification_status
    - loan_status
    - purpose
    - zip_code
    - addr_state
    - initial_list_status
    - application_type
- **Datetime Objects**
    - issue_d
    - earliest_cr_line
    - last_pymnt_d
    - last_credit_pull_d
- **Potential Float, Int, or Boolean**
    - term (Int)
    - int_rate (Float)
    - pymnt_plan (Boolean)
    - revol_util (Float)
- **No Alterations**
    - emp_title
    - title

We have prepared the lists below to process this information accordingly.

In [10]:
cdte_categorisation = {'grade':'category',
                       'sub_grade':'category',
                       'emp_length':'category',
                       'home_ownership':'category',
                       'verification_status':'category',
                       'loan_status':'category',
                       'purpose':'category',
                       'zip_code':'category',
                       'addr_state':'category',
                       'initial_list_status':'category',
                       'application_type':'category'}
cdte_datetime = ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']
cdte_float = ['int_rate', 'revol_util']
cdte_int = ['term']
cdte_boolean = ['pymnt_plan']

invesigation_load.dtypes # This is printed so we can see the

term                   str
int_rate               str
grade                  str
sub_grade              str
emp_title              str
emp_length             str
home_ownership         str
verification_status    str
issue_d                str
loan_status            str
pymnt_plan             str
purpose                str
title                  str
zip_code               str
addr_state             str
earliest_cr_line       str
revol_util             str
initial_list_status    str
last_pymnt_d           str
last_credit_pull_d     str
application_type       str
dtype: object

In [11]:
# When loading the data, we will parse the datetime objects and categorize the columns accordingly.

loans = pd.read_csv('loans_2007.csv',
                    chunksize=chunk_size,
                    skiprows=skip,
                    date_format='%b-%Y',
                    parse_dates=cdte_datetime,
                    dtype = cdte_categorisation
)

chunk_size_MB = []

# We will now iterate through our chunks and update the last few columns with minor data alterations.

for chunk in loans:

    for col in cdte_float:
        cleaned = chunk[col].str.rstrip("%")
        chunk[col] = pd.to_numeric(cleaned)

    for col in cdte_boolean:
        chunk[col] = chunk[col].map({'y': True, 'n': False}).astype('bool')

    for col in cdte_int:
        cleaned = chunk[col].str.lstrip(" ").str.rstrip(" months")
        chunk[col] = pd.to_numeric(cleaned)

    chunk_size_MB.append(chunk.memory_usage(deep=True).sum()/1024 ** 2)

print(f"The new average memory usage of each loaded chunk is {sum(chunk_size_MB)/len(chunk_size_MB):.2f} MB")
print(f"The new total memory is {sum(chunk_size_MB):.2f} MB")

new_memory_footprint = sum(chunk_size_MB)
print(f"The ratio of the new memory footprint to the original memory footprint is {new_memory_footprint/original_memory_footprint:.2f}, which represents a {original_memory_footprint/new_memory_footprint:.2f}x saving in memory usage")

The new average memory usage of each loaded chunk is 3.01 MB
The new total memory is 15.07 MB
The ratio of the new memory footprint to the original memory footprint is 0.64, which represents a 1.56x saving in memory usage


We can see the effect of our data cleaning on memory usage. By stripping unnecessary characters and converting data types, we were able to reduce the memory footprint of loaded data by approximately 3x. This is a significant improvement and can help us handle larger datasets more efficiently. We will build a simple cleaner function and load the "loans data" using the cleaner, which will now take up considerably less RAM for storage. In a more ideal scenario, we would pre-define our "categories", as the `pd.concat()` method reverts some `category` dtypes back to `str`, but for the purpose of this demonstration, the desired effect has been achieved.

In [12]:
def string_cleaner(chunk):
    for col in cdte_float:
        cleaned = chunk[col].str.rstrip("%")
        chunk[col] = pd.to_numeric(cleaned)
    for col in cdte_boolean:
        chunk[col] = chunk[col].map({'y': True, 'n': False}).astype('bool')
    for col in cdte_int:
        cleaned = chunk[col].str.lstrip(" ").str.rstrip(" months")
        chunk[col] = pd.to_numeric(cleaned)
    return chunk

loans = pd.read_csv('loans_2007.csv',
                    chunksize=chunk_size,
                    skiprows=skip,
                    date_format='%b-%Y',
                    parse_dates=cdte_datetime,
                    dtype = cdte_categorisation
)

loans_df = pd.concat(string_cleaner(chunk) for chunk in loans)

## Optimising Numeric Columns

Using the cleaner function, we have successfully converted string columns to numeric types where appropriate. However, we can further optimize our data by converting numeric columns to more efficient data types. This will help reduce memory usage and improve performance. We will utilise the list of columns we previously identified under the **Float Columns Candidates for Integer Conversion** section to add to our previous code.

In [13]:
def string_cleaner(chunk):
    for col in cdte_float:
        chunk[col] = pd.to_numeric(chunk[col].str.rstrip("%"), errors='coerce')

    for col in cdte_boolean:
        mapped = chunk[col].str.strip().str.lower().map({'y': True, 'n': False})
        chunk[col] = mapped.astype('bool') if mapped.notna().all() else mapped.astype('boolean')

    for col in cdte_int:
        cleaned = chunk[col].str.strip().str.replace(' months', '', regex=False)
        chunk[col] = pd.to_numeric(cleaned, errors='coerce')

    return chunk

def profile_numeric(chunks):
    stats = {}

    for chunk in chunks:
        chunk = string_cleaner(chunk)

        for col in chunk.select_dtypes(include=['float']).columns:
            entry = stats.setdefault(col, {
                'all_whole': True, 'has_na': False,
                'min': np.inf, 'max': -np.inf,
            })

            entry['has_na'] |= bool(chunk[col].isna().any())

            non_null = chunk[col].dropna()
            if not len(non_null):
                continue

            entry['all_whole'] &= bool((non_null % 1 == 0).all())
            entry['min'] = min(entry['min'], float(non_null.min()))
            entry['max'] = max(entry['max'], float(non_null.max()))

    return stats


def build_plan(stats):
    plan = {}

    for col, s in stats.items():
        if s['min'] == np.inf:
            continue

        bounds = pd.Series([s['min'], s['max']])
        kind = 'integer' if (s['all_whole'] and not s['has_na']) else 'float'
        plan[col] = pd.to_numeric(bounds, downcast=kind).dtype

    return plan


def numeric_cleaner(chunk, plan):
    return chunk.astype({c: d for c, d in plan.items() if c in chunk.columns})

def load_loans():
    stats = profile_numeric(pd.read_csv('loans_2007.csv',
                    chunksize=chunk_size,
                    skiprows=skip,
                    date_format='%b-%Y',
                    parse_dates=cdte_datetime,
                    dtype = cdte_categorisation
    ))
    plan = build_plan(stats)

    def clean(chunk):
        chunk = string_cleaner(chunk)
        return numeric_cleaner(chunk, plan)

    chunks = pd.read_csv('loans_2007.csv',
                    chunksize=chunk_size,
                    skiprows=skip,
                    date_format='%b-%Y',
                    parse_dates=cdte_datetime,
                    dtype = cdte_categorisation
    )
    df = pd.concat((clean(c) for c in chunks), ignore_index=True)

    return df

loans_df = load_loans()
loans_df.head(10)

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,1077501,1296599,5000,5000,4975.0,36,10.650000,162.869995,B,B2,...,171.62,2016-06-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
1,1077430,1314167,2500,2500,2500.0,60,15.270000,59.830002,C,C4,...,119.66,2013-09-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
2,1077175,1313524,2400,2400,2400.0,36,15.960000,84.330002,C,C5,...,649.91,2016-06-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
3,1076863,1277178,10000,10000,10000.0,36,13.490000,339.309998,C,C1,...,357.48,2016-04-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
4,1075358,1311748,3000,3000,3000.0,60,12.690000,67.790001,B,B5,...,67.79,2016-06-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
5,1075269,1311441,5000,5000,5000.0,36,7.900000,156.460007,A,A4,...,161.03,2016-01-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
6,1069639,1304742,7000,7000,7000.0,60,15.960000,170.080002,C,C5,...,1313.76,2016-05-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
7,1072053,1288686,3000,3000,3000.0,36,18.639999,109.430000,E,E1,...,111.34,2014-12-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
8,1071795,1306957,5600,5600,5600.0,60,21.280001,152.389999,F,F2,...,152.39,2012-08-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
9,1071570,1306721,5375,5375,5350.0,60,12.690000,121.449997,B,B5,...,121.45,2013-03-01,0.0,1,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
